In [ ]:
import pandas as pd
import numpy as np
import os
import tabulate
import matplotlib.pyplot as plt
from scipy import stats

path10 = 'D:/UCSF_postdoc_topic/ESI_/new_data_acquisition/ISTD_RT_manualcheck2.csv'  #other mixtures mzlist with updated conc in ug/ml

# =====================================================================
# 1. Build the Calibration Models (Keyed by DTXSID)
# =====================================================================
calib_mzrt = pd.read_csv(r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\calibration_curve_QAC_mzrt_table.csv')
calib_mzrt['compound_name'] = calib_mzrt['Name']
print(tabulate.tabulate(calib_mzrt, headers='keys', tablefmt='psql'))
#peak area from msdial processed peak area.

# Step 1: Load the Data
# Load the raw LC-MS data file
peakareadata = r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\Copy of 04022026\Area_1_2026_04_03_14_54_13.csv'
peakarea = pd.read_csv(peakareadata)
feature_data = peakarea.copy()
print(feature_data.shape)

+----+---------------------------------------------+-----------+----------------+---------+--------+---------------------------------------------+
|    | Name                                        | CAS       | DTXSID         |      MZ |     RT | compound_name                               |
|----+---------------------------------------------+-----------+----------------+---------+--------+---------------------------------------------|
|  0 | Choline chloride                            | 67-48-1   | DTXSID4020325  | 104.108 |  0.783 | Choline chloride                            |
|  1 | (3-Carboxypropyl)trimethylammonium chloride | 6249-56-5 | DTXSID00978073 | 146.118 |  0.766 | (3-Carboxypropyl)trimethylammonium chloride |
|  2 | C12-ATMAC                                   | 112-00-5  | DTXSID1026900  | 228.269 | 11.187 | C12-ATMAC                                   |
|  3 | C12-BAC                                     | 139-07-1  | DTXSID2040787  | 304.3   | 12.446 | C12-BAC          

In [44]:
# Step 2: Define sample groups
sample_name1 = ['CAL0','CAL1', 'CAL2', 'CAL3', 'CAL4', 'CAL5', 'CAL6', 'CAL7']

# Find blank sample columns
blank_samples = [col for col in feature_data.columns if col.startswith('IB')]

# Step 3: Average replicates for each calibration level
newcolumns = []

for sample in sample_name1:
    sample_conc = [f'{sample}_r1', f'{sample}_r2', f'{sample}_r3']

    # Check that all expected replicate columns exist
    missing_cols = [col for col in sample_conc if col not in feature_data.columns]
    if missing_cols:
        print(f'Warning: missing replicate columns for {sample}: {missing_cols}')
        continue

    feature_data[sample] = feature_data[sample_conc].mean(axis=1)
    newcolumns.append(sample)

# Step 4: Subset required columns
required_metadata = ['Average Rt(min)', 'Alignment ID', 'Average Mz']
required_cols = required_metadata + newcolumns + blank_samples

# step 5: remove rows with all misisng vaue
missing_required = [col for col in required_cols if col not in feature_data.columns]
if missing_required:
    raise KeyError(f'Missing required columns: {missing_required}')

feature_data2 = feature_data[required_cols].copy()
feature_data_istd = feature_data2.copy()
feature_data_istd['mean'] = feature_data_istd[newcolumns].mean(axis=1)

# feature_data2[newcolumns] = feature_data2[newcolumns].sub(feature_data2['CAL0'], axis=0)
feature_data2 = feature_data2.drop(columns=blank_samples)
# step 6: remove rows with all na or negative value
feature_data2 = feature_data2[(feature_data2[newcolumns] > 0).any(axis=1)]

#step 7: get max value, remove row with max value <=5000
feature_data2['median'] = feature_data2[newcolumns].median(axis = 1)
feature_data2['mean'] = feature_data2[newcolumns].mean(axis = 1)
feature_data2 = feature_data2[(feature_data2['median'] >= 5000)]

#filter in rows with CAL1 >=5000
# feature_data2 = feature_data2[(feature_data2['CAL1']>=1000)]

print(f'Number of rows after blank removal: {feature_data2.shape}')
print(f'Number of rows after blank removal: {feature_data_istd.shape}')
print(tabulate.tabulate(feature_data2.head(), headers='keys', tablefmt='psql', showindex=False))

Number of rows after blank removal: (5629, 13)
Number of rows after blank removal: (7873, 14)
+-------------------+----------------+--------------+-----------------+-----------------+----------------+-----------------+-----------------+-----------------+------------------+------------------+-----------------+-----------------+
|   Average Rt(min) |   Alignment ID |   Average Mz |            CAL0 |            CAL1 |           CAL2 |            CAL3 |            CAL4 |            CAL5 |             CAL6 |             CAL7 |          median |            mean |
|-------------------+----------------+--------------+-----------------+-----------------+----------------+-----------------+-----------------+-----------------+------------------+------------------+-----------------+-----------------|
|             1.169 |              0 |      100.112 |  1055.33        |  1439.67        |  3469.33       |  9076.67        | 20779.3         | 30327.3         |  87766           | 175144           | 14

In [50]:
# Step 3: perform mz/RT match with mzlist and RTlist for each sample
# Extract the best matched feature using maximum CAL peak area

import numpy as np
import pandas as pd
import tabulate


def ppm(mz1, mz2):
    return abs(mz1 - mz2) / mz1 * 1e6

def rt_match(rt1, rt2):
    return abs(rt1 - rt2)

def findpeak(featdata, concta, mstol, rttol):
    """
    Match QAC features to targeted compounds using m/z and RT tolerance.

    If multiple features are matched to one targeted compound:
    - for each CAL column, extract the maximum peak area across matched features
    - keep Average Mz and Average Rt(min) from the first matched feature
    - recalculate mean and median after final CAL peak areas are assigned
    """

    mstol = float(mstol)
    rttol = float(rttol)

    cal_cols = ['CAL0', 'CAL1', 'CAL2', 'CAL3', 'CAL4', 'CAL5', 'CAL6', 'CAL7']

    required_feat_cols = ['Average Mz', 'Average Rt(min)'] + cal_cols
    required_concta_cols = ['MZ', 'RT']

    missing_feat = [c for c in required_feat_cols if c not in featdata.columns]
    missing_concta = [c for c in required_concta_cols if c not in concta.columns]

    if missing_feat:
        raise KeyError(f"featdata is missing required columns: {missing_feat}")
    if missing_concta:
        raise KeyError(f"concta is missing required columns: {missing_concta}")

    temp_feat = featdata.copy()
    temp_concta = concta.copy()

    temp_feat['Average Mz'] = pd.to_numeric(temp_feat['Average Mz'], errors='coerce')
    temp_feat['Average Rt(min)'] = pd.to_numeric(temp_feat['Average Rt(min)'], errors='coerce')

    for col in cal_cols:
        temp_feat[col] = pd.to_numeric(temp_feat[col], errors='coerce')

    temp_concta['MZ'] = pd.to_numeric(temp_concta['MZ'], errors='coerce')
    temp_concta['RT'] = pd.to_numeric(temp_concta['RT'], errors='coerce')

    final_rows = []

    for compound_index, compound in temp_concta.iterrows():
        target_mz = compound['MZ']
        target_rt = compound['RT']

        if pd.isna(target_mz) or pd.isna(target_rt):
            continue

        mz_error = abs(temp_feat['Average Mz'] - target_mz) / target_mz * 1e6
        rt_error = abs(temp_feat['Average Rt(min)'] - target_rt)

        matched_rows = temp_feat[
            (mz_error <= mstol) &
            (rt_error <= rttol)
        ].copy()

        if matched_rows.empty:
            continue

        # First matched feature only provides representative metadata
        first_match = matched_rows.iloc[0].copy()
        final_row = first_match.copy()

        # Column-wise max peak area extraction
        for col in cal_cols:
            col_values = pd.to_numeric(matched_rows[col], errors='coerce')
            final_row[col] = col_values.max(skipna=True)

        # Recalculate mean and median from the final extracted CAL values
        final_cal_values = pd.to_numeric(final_row[cal_cols], errors='coerce')

        final_row['mean'] = final_cal_values.mean(skipna=True)
        final_row['median'] = final_cal_values.median(skipna=True)

        # Add target information
        final_row['compound_index'] = compound_index

        if 'Name' in temp_concta.columns:
            final_row['compound_name'] = compound['Name']

        final_row['target_MZ'] = target_mz
        final_row['target_RT'] = target_rt

        final_row['ppm_error'] = abs(first_match['Average Mz'] - target_mz) / target_mz * 1e6
        final_row['RT_error'] = abs(first_match['Average Rt(min)'] - target_rt)

        final_row['n_matched_features'] = matched_rows.shape[0]
        final_row['peak_area_extraction_method'] = 'column_wise_max_CAL_area'

        final_rows.append(final_row)

    if not final_rows:
        return pd.DataFrame()

    return pd.DataFrame(final_rows).reset_index(drop=True)


def findpeak2(featdata, concta, mstol, rttol, sample_cols=None):
    """
    Match ISTD features using m/z and RT tolerance.

    If multiple features are matched to one target:
    - for each sample/CAL column, extract the maximum peak area across matched features
    - keep Average Mz and Average Rt(min) from the first matched feature
    - output one final row per target
    """

    mstol = float(mstol)
    rttol = float(rttol)

    if sample_cols is None:
        sample_cols = [
            c for c in featdata.columns
            if str(c).startswith("CAL")
        ]

    temp_feat = featdata.copy()
    temp_concta = concta.copy()

    temp_feat['Average Mz'] = pd.to_numeric(temp_feat['Average Mz'], errors='coerce')
    temp_feat['Average Rt(min)'] = pd.to_numeric(temp_feat['Average Rt(min)'], errors='coerce')

    for col in sample_cols:
        temp_feat[col] = pd.to_numeric(temp_feat[col], errors='coerce')

    temp_concta['MZ'] = pd.to_numeric(temp_concta['MZ'], errors='coerce')
    temp_concta['RT'] = pd.to_numeric(temp_concta['RT'], errors='coerce')

    final_rows = []

    for compound_index, compound in temp_concta.iterrows():
        target_mz = compound['MZ']
        target_rt = compound['RT']

        if pd.isna(target_mz) or pd.isna(target_rt):
            continue

        mz_error = abs(temp_feat['Average Mz'] - target_mz) / target_mz * 1e6
        rt_error = abs(temp_feat['Average Rt(min)'] - target_rt)

        matched_rows = temp_feat[
            (mz_error <= mstol) &
            (rt_error <= rttol)
        ].copy()

        if matched_rows.empty:
            continue

        # First matched feature provides representative m/z and RT
        first_match = matched_rows.iloc[0].copy()

        # Column-wise max peak area extraction
        final_peak_areas = matched_rows[sample_cols].max(axis=0, skipna=True)

        final_row = first_match.copy()

        for col in sample_cols:
            final_row[col] = final_peak_areas[col]

        final_row['compound_index'] = compound_index

        if 'Name' in temp_concta.columns:
            final_row['compound_name'] = compound['Name']

        final_row['target_MZ'] = target_mz
        final_row['target_RT'] = target_rt
        final_row['ppm_error'] = abs(first_match['Average Mz'] - target_mz) / target_mz * 1e6
        final_row['RT_error'] = abs(first_match['Average Rt(min)'] - target_rt)
        final_row['n_matched_features'] = matched_rows.shape[0]
        final_row['peak_area_extraction_method'] = 'column_wise_max_area'

        final_rows.append(final_row)

    if not final_rows:
        return pd.DataFrame()

    return pd.DataFrame(final_rows).reset_index(drop=True)


# ==================================================
# Perform mz/RT search for internal standards
# ==================================================
mstol = 10.0  # ppm
rttol = 0.5   # min

ISTD_rt = calib_mzrt[calib_mzrt['CAS'] == 'ISTD']
ISpeak_cali = findpeak2(
    feature_data_istd,
    ISTD_rt,
    mstol,
    rttol
)

print(f'number of matched rows: {ISpeak_cali.shape}')
print(tabulate.tabulate(ISpeak_cali, headers='keys', tablefmt='psql'))


# ==================================================
# Perform mz/RT search for QAC calibration compounds
# Selection is based on maximum CAL peak area
# ==================================================
mstol = 25.0  # ppm
rttol = 0.5   # min
QAC_rt = calib_mzrt[calib_mzrt['CAS'] != 'ISTD']

QAC_cali = findpeak(
    feature_data2,
    QAC_rt,
    mstol,
    rttol
)

print(f'number of matched rows: {QAC_cali.shape}')
print(tabulate.tabulate(QAC_cali, headers='keys', tablefmt='psql'))

number of matched rows: (2, 22)
+----+-------------------+----------------+--------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------+-------+-------------+------------------+-----------------+-------------+-------------+-------------+------------+----------------------+-------------------------------+
|    |   Average Rt(min) |   Alignment ID |   Average Mz |        CAL0 |        CAL1 |        CAL2 |        CAL3 |        CAL4 |        CAL5 |        CAL6 |        CAL7 |   IB1 |   IB2 |        mean |   compound_index | compound_name   |   target_MZ |   target_RT |   ppm_error |   RT_error |   n_matched_features | peak_area_extraction_method   |
|----+-------------------+----------------+--------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------+-------+-------------+------------------+-----------------+-------------+-------------+-------

In [ ]:
#output the data table to get the method detection limit of each compounds
QAC_cali.to_csv(r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\Copy of 04022026\peak_area_calibrationcurve_forMDL_27Apr2026.csv')

In [ ]:
def build_calibration_curves(
    QAC_cali,
    ISpeak_cali,
    compound_col='compound_name',
    is_name='D15-TPP',
    output_root=r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\Copy of 04022026'
):
    """
    Build linear calibration curves.

    Steps
    -----
    1. Blank subtraction using CAL0
    2. Normalize by internal standard
    3. Fit linear calibration curve: y = slope * x + intercept
    4. Save calibration plots and model summary
    """

    cal0_col = 'CAL0'
    cal_cols = ['CAL1', 'CAL2', 'CAL3', 'CAL4', 'CAL5', 'CAL6']
    all_cal_cols = [cal0_col] + cal_cols

    conc_map = {
        'CAL1': 1,
        'CAL2': 5,
        'CAL3': 10,
        'CAL4': 25,
        'CAL5': 50,
        'CAL6': 100
    }

    compound_point_rules = {
        'Choline chloride': ['CAL2', 'CAL3', 'CAL4', 'CAL5'],
        '(3-Carboxypropyl)trimethylammonium chloride': ['CAL1', 'CAL2', 'CAL3', 'CAL4'],
        'C12-BAC':['CAL2', 'CAL3', 'CAL4', 'CAL5', 'CAL6'],
        'C14-BAC':['CAL2', 'CAL3', 'CAL4', 'CAL5', 'CAL6'],
        'C16-ATMAC':['CAL1','CAL2', 'CAL3', 'CAL4', 'CAL6']
    }

    plot_dir = os.path.join(output_root, 'calibration_curve_plot')
    os.makedirs(plot_dir, exist_ok=True)

    qac = QAC_cali.copy()
    ispk = ISpeak_cali.copy()

    # Check required columns
    required_cols = [compound_col] + all_cal_cols

    missing_qac = [c for c in required_cols if c not in qac.columns]
    missing_ispk = [c for c in required_cols if c not in ispk.columns]

    if missing_qac:
        raise KeyError(f"QAC_cali is missing columns: {missing_qac}")
    if missing_ispk:
        raise KeyError(f"ISpeak_cali is missing columns: {missing_ispk}")

    # Convert calibration columns to numeric
    for c in all_cal_cols:
        qac[c] = pd.to_numeric(qac[c], errors='coerce')
        ispk[c] = pd.to_numeric(ispk[c], errors='coerce')

    # Blank subtraction
    qac[cal_cols] = qac[cal_cols].sub(qac[cal0_col], axis=0)
    qac[cal_cols] = qac[cal_cols].clip(lower=0)

    # Internal standard normalization
    is_row = ispk.loc[ispk[compound_col] == is_name]

    if is_row.empty:
        raise ValueError(f"Internal standard '{is_name}' not found.")

    is_values = is_row.iloc[0][cal_cols].astype(float)

    normalized_df = qac[[compound_col] + cal_cols].copy()
    normalized_df[cal_cols] = normalized_df[cal_cols].div(is_values, axis=1)

    results = []

    for _, row in normalized_df.iterrows():
        compound_name = row[compound_col]

        used_cols = compound_point_rules.get(compound_name, cal_cols)

        x = np.array([conc_map[c] for c in used_cols], dtype=float)
        y = row[used_cols].to_numpy(dtype=float)

        valid = np.isfinite(x) & np.isfinite(y)
        x = x[valid]
        y = y[valid]
        used_cols_valid = list(np.array(used_cols)[valid])

        if len(x) < 2:
            results.append({
                'compound_name': compound_name,
                'model': None,
                'r2': np.nan,
                'slope': np.nan,
                'intercept': np.nan,
                'equation': None,
                'used_points': ','.join(used_cols_valid)
            })
            continue

        # Linear model
        lin = stats.linregress(x, y)
        slope = lin.slope
        intercept = lin.intercept
        r2 = lin.rvalue ** 2

        equation = f"y = {slope:.6g}x + {intercept:.6g}"

        results.append({
            'compound_name': compound_name,
            'model': 'linear',
            'r2': r2,
            'slope': slope,
            'intercept': intercept,
            'equation': equation,
            'used_points': ','.join(used_cols_valid)
        })

        # Plot calibration curve
        fig, ax = plt.subplots(figsize=(6, 5))

        ax.scatter(x, y, s=45)

        x_line = np.linspace(np.min(x), np.max(x), 300)
        y_line = slope * x_line + intercept

        ax.plot(x_line, y_line, linewidth=2)

        ax.set_xlabel('Concentration (ppb)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Normalized peak area', fontsize=12, fontweight='bold')
        ax.set_title(compound_name, fontsize=13, fontweight='bold')

        txt = f"{equation}\nR² = {r2:.4f}"
        ax.text(
            0.05, 0.95, txt,
            transform=ax.transAxes,
            va='top',
            ha='left',
            fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='both', labelsize=10)

        plt.tight_layout()

        safe_name = "".join(
            ch if ch.isalnum() or ch in "._-" else "_"
            for ch in str(compound_name)
        )

        save_path = os.path.join(plot_dir, f"{safe_name}_linear_calibration_curve.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

    model_summary = pd.DataFrame(results)

    summary_path = os.path.join(output_root, 'linear_calibration_curve_model_summary.csv')
    normalized_path = os.path.join(output_root, 'normalized_calibration_peak_area.csv')

    model_summary.to_csv(summary_path, index=False)
    normalized_df.to_csv(normalized_path, index=False)

    return normalized_df, model_summary


normalized_df, model_summary = build_calibration_curves(
    QAC_cali=QAC_cali,
    ISpeak_cali=ISpeak_cali,
    compound_col='compound_name',
    is_name='D15-TPP',
    output_root=r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\Copy of 04022026'
)

print(model_summary.head())

# -----------------------------
# Step 1: merge calibration info
# -----------------------------
cal_curve = pd.merge(
    model_summary,
    calib_mzrt[['compound_name', 'DTXSID','RT']],
    on='compound_name',
    how='left'
)

print(tabulate.tabulate(cal_curve, headers='keys', tablefmt='psql', showindex=False))
cal_curve.to_csv(r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\calibration_curve_QAC_modelsummary.csv')

                                 compound_name   model        r2     slope  \
0                             Choline chloride  linear  0.972443  0.000033   
1  (3-Carboxypropyl)trimethylammonium chloride  linear  0.983562  0.007402   
2                                    C12-ATMAC  linear  0.996786  0.031192   
3                                      C12-BAC  linear  0.990087  0.021553   
4                                      C16-BAC  linear  0.998334  0.010903   

   intercept                        equation                    used_points  
0  -0.000048  y = 3.2941e-05x + -4.76781e-05            CAL2,CAL3,CAL4,CAL5  
1  -0.003628   y = 0.00740248x + -0.00362805            CAL1,CAL2,CAL3,CAL4  
2   0.080503      y = 0.0311918x + 0.0805032  CAL1,CAL2,CAL3,CAL4,CAL5,CAL6  
3   0.169341       y = 0.0215529x + 0.169341       CAL2,CAL3,CAL4,CAL5,CAL6  
4  -0.001697    y = 0.0109034x + -0.00169677  CAL1,CAL2,CAL3,CAL4,CAL5,CAL6  


In [33]:
pos_filtered_final = pd.read_csv(r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\8QACs_normalized_peak_area_withISTD_25042026.csv').copy()

In [58]:
# -----------------------------
# Step 3: estimate concentrations
# -----------------------------
sample_cols = [col for col in pos_filtered_final.columns if col.startswith('BH')]

if not sample_cols:
    raise ValueError("No sample columns starting with 'BH' were found.")

# merge calibration parameters into real sample peak table
pos_est = pd.merge(
    pos_filtered_final,
    cal_curve,
    on='DTXSID',
    how='left',
    suffixes=('', '_cal')
)

def solve_concentration_from_row(row, peak_area):
    """
    Estimate concentration from one peak area using calibration parameters in one row.
    Returns NaN if peak_area is NaN or if model is invalid.
    """
    if pd.isna(peak_area):
        return np.nan

    model = row['model']

    if pd.isna(model):
        return np.nan

    # linear model: y = slope*x + intercept
    if model == 'linear':
        slope = row['slope']
        intercept = row['intercept']

        if pd.isna(slope) or slope == 0 or pd.isna(intercept):
            return np.nan

        x = (peak_area - intercept) / slope

        # optional: do not allow negative estimated concentration
        return x if x >= 0 else np.nan

    # quadratic model: y = a*x^2 + b*x + c
    elif model == 'quadratic':
        a = row['quad_a']
        b = row['quad_b']
        c = row['quad_c']

        if pd.isna(a) or pd.isna(b) or pd.isna(c):
            return np.nan

        # solve a*x^2 + b*x + c - peak_area = 0
        coeffs = [a, b, c - peak_area]
        roots = np.roots(coeffs)

        # keep only real roots
        real_roots = roots[np.isreal(roots)].real

        # keep only non-negative roots
        valid_roots = real_roots[real_roots >= 0]

        if len(valid_roots) == 0:
            return np.nan

        # choose the smaller non-negative root by default
        # this is often the more conservative choice for calibration curves
        return np.min(valid_roots)

    else:
        return np.nan


# estimate concentration for each BH column
for col in sample_cols:
    conc_col = f'{col}_conc'
    pos_est[conc_col] = pos_est.apply(lambda row: solve_concentration_from_row(row, row[col]), axis=1)

print(tabulate.tabulate(
    pos_est[['DTXSID', 'compound_name'] + sample_cols[:3] + [f'{c}_conc' for c in sample_cols[:3]]].head(),
    headers='keys',
    tablefmt='psql',
    showindex=False
))

C:\Users\yangj\AppData\Local\Temp\ipykernel_26248\4055332413.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pos_est[conc_col] = pos_est.apply(lambda row: solve_concentration_from_row(row, row[col]), axis=1)
C:\Users\yangj\AppData\Local\Temp\ipykernel_26248\4055332413.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pos_est[conc_col] = pos_est.apply(lambda row: solve_concentration_from_row(row, row[col]), axis=1)
C:\Users\yangj\AppData\Local\Temp\ipykernel_26248\4055332413.py:77: PerformanceWarning: DataFrame is highly 

+----------------+---------------------------------------------+-------------+-------------+-------------+----------------+----------------+----------------+
| DTXSID         | compound_name                               |     BH04254 |     BH04255 |     BH04256 |   BH04254_conc |   BH04255_conc |   BH04256_conc |
|----------------+---------------------------------------------+-------------+-------------+-------------+----------------+----------------+----------------|
| DTXSID00978073 | (3-Carboxypropyl)trimethylammonium chloride | 0.0180364   | 0.0369571   | 0.0760734   |       2.92665  |       5.48264  |      10.7669   |
| DTXSID1026900  | C12-ATMAC                                   | 5.68445e-05 | 0.00439159  | 0.00409156  |     nan        |     nan        |     nan        |
| DTXSID2025139  | C18-BAC                                     | 0           | 0.000103415 | 0.000143965 |     nan        |     nan        |     nan        |
| DTXSID2040787  | C12-BAC                          

In [ ]:
# ==================================================
# statistic analysis of estimated concentration of confirmed QAC compounds
# ==================================================
df = pos_est.copy()

# ==================================================
# Define MDL and LOQ thresholds for QAC compounds
# ==================================================
mdl_dict = {
    "DTXSID4020325":3.282,
    "DTXSID00978073":0.0235,
    "DTXSID1026900": 0.025,
    "DTXSID2040787": 0.171,
    "DTXSID3041665": 0.445,
    "DTXSID4046606": 0.577,
    "DTXSID2025139": 0.0138,
    "DTXSID6026901": 0.863,
}

LOQ_dict = {
    "DTXSID4020325": 10.941,
    "DTXSID00978073": 0.0783,
    "DTXSID1026900": 0.0834,
    "DTXSID2040787": 0.569,
    "DTXSID3041665": 1.483,
    "DTXSID4046606": 1.923,
    "DTXSID2025139": 0.0459,
    "DTXSID6026901": 2.876,
}

# ==================================================
# Volume correction factor
# Final analyte volume = 150 uL
# Original serum volume = 250 uL
# Factor = 150 / 250 = 0.6
# Applied only AFTER MDL/LOQ detection filtering
# ==================================================
VOLUME_CORRECTION_FACTOR = 0.6

# ==================================================
# Step 1. Data reduction
# Retain compound annotation columns and concentration columns
# ==================================================
annotation_cols = [
    "compound_name",
    "DTXSID",
    "RT",
    "Average Rt(min)",
    "Average Mz"
]

annotation_cols = [c for c in annotation_cols if c in df.columns]

if "DTXSID" not in annotation_cols:
    raise KeyError("DTXSID column is required but was not found in df.")

# Select concentration sample columns like BH04254_conc
conc_cols_original = [
    c for c in df.columns
    if str(c).startswith("BH") and str(c).endswith("_conc")
]

if not conc_cols_original:
    raise ValueError("No concentration columns found. Expected columns like BH04254_conc.")

concentration_table_original = df[annotation_cols + conc_cols_original].copy()

# ==================================================
# Step 2. Rename BH04254_conc to BH04254
# ==================================================
rename_conc_cols = {
    c: c.replace("_conc", "")
    for c in conc_cols_original
}

concentration_table_original = concentration_table_original.rename(columns=rename_conc_cols)
conc_cols = list(rename_conc_cols.values())

# Convert concentration columns to numeric
for col in conc_cols:
    concentration_table_original[col] = pd.to_numeric(
        concentration_table_original[col],
        errors="coerce"
    )

print(f"Original concentration table shape before filtering: {concentration_table_original.shape}")
display(concentration_table_original.head())


# ==================================================
# Function: threshold filtering and summary
# Detection frequency is calculated using original concentration
# Then output concentration table is volume-corrected
# ==================================================
def filter_concentration_by_threshold(
    concentration_table_original,
    conc_cols,
    threshold_dict,
    threshold_label="MDL",
    volume_correction_factor=0.6
):
    """
    For each DTXSID:
    1. Use original uncorrected concentration for MDL/LOQ filtering.
    2. Count values below threshold, excluding existing NaN.
    3. Replace values below threshold with NaN.
    4. Calculate detection frequency using original concentration.
    5. Apply volume correction to the final filtered concentration table.
    6. Summarize volume-corrected concentrations after filtering.

    Notes
    -----
    MDL/LOQ thresholds are compared with original estimated concentrations.
    Volume correction is applied only after detection classification.
    """

    # --------------------------------------------------
    # Summary BEFORE filtering, original concentration
    # --------------------------------------------------
    before_summary_records = []

    for idx, row in concentration_table_original.iterrows():
        dtxsid = str(row["DTXSID"])
        values_original = pd.to_numeric(row[conc_cols], errors="coerce")

        n_nan_before = int(values_original.isna().sum())
        n_non_nan_before = int(values_original.notna().sum())

        if dtxsid in threshold_dict:
            threshold = threshold_dict[dtxsid]
            below_threshold_mask = values_original.notna() & (values_original < threshold)
            n_below_threshold = int(below_threshold_mask.sum())
        else:
            threshold = np.nan
            n_below_threshold = np.nan

        before_summary_records.append({
            "compound_name": row["compound_name"] if "compound_name" in concentration_table_original.columns else np.nan,
            "DTXSID": dtxsid,
            "RT": row["RT"] if "RT" in concentration_table_original.columns else np.nan,
            "Average Rt(min)": row["Average Rt(min)"] if "Average Rt(min)" in concentration_table_original.columns else np.nan,
            "Average Mz": row["Average Mz"] if "Average Mz" in concentration_table_original.columns else np.nan,
            f"{threshold_label}_threshold_original_conc": threshold,

            f"min_before_{threshold_label}_filtering_original_conc": values_original.min(skipna=True),
            f"25th_percentile_before_{threshold_label}_filtering_original_conc": values_original.quantile(0.25),
            f"median_before_{threshold_label}_filtering_original_conc": values_original.median(skipna=True),
            f"mean_before_{threshold_label}_filtering_original_conc": values_original.mean(skipna=True),
            f"sd_before_{threshold_label}_filtering_original_conc": values_original.std(skipna=True, ddof=1),
            f"75th_percentile_before_{threshold_label}_filtering_original_conc": values_original.quantile(0.75),
            f"max_before_{threshold_label}_filtering_original_conc": values_original.max(skipna=True),

            f"n_nan_before_{threshold_label}_filtering": n_nan_before,
            f"n_values_below_{threshold_label}_excluding_nan": n_below_threshold,
            "n_total_samples": len(conc_cols),
            f"n_non_nan_before_{threshold_label}_filtering": n_non_nan_before
        })

    concentration_summary_before_filtering = pd.DataFrame(before_summary_records)

    # --------------------------------------------------
    # Apply filtering using original concentration
    # --------------------------------------------------
    concentration_table_filtered_original = concentration_table_original.copy()

    for dtxsid, threshold in threshold_dict.items():
        row_mask = concentration_table_filtered_original["DTXSID"].astype(str).eq(dtxsid)

        if row_mask.sum() == 0:
            print(f"Warning: {dtxsid} not found in concentration table for {threshold_label} filtering.")
            continue

        values_original = concentration_table_filtered_original.loc[row_mask, conc_cols]
        values_original = values_original.apply(pd.to_numeric, errors="coerce")

        below_threshold_mask = values_original.notna() & (values_original < threshold)

        concentration_table_filtered_original.loc[row_mask, conc_cols] = values_original.mask(
            below_threshold_mask,
            np.nan
        )

    # --------------------------------------------------
    # Apply volume correction AFTER filtering
    # --------------------------------------------------
    concentration_table_filtered_corrected = concentration_table_filtered_original.copy()
    concentration_table_filtered_corrected[conc_cols] = (
        concentration_table_filtered_corrected[conc_cols] * volume_correction_factor
    )

    # --------------------------------------------------
    # Summary AFTER filtering, volume-corrected concentration
    # --------------------------------------------------
    after_summary_records = []

    for idx, row in concentration_table_filtered_corrected.iterrows():
        dtxsid = str(row["DTXSID"])
        values_corrected = pd.to_numeric(row[conc_cols], errors="coerce")

        n_nan_after = int(values_corrected.isna().sum())
        n_detected_after = int(values_corrected.notna().sum())

        after_summary_records.append({
            "DTXSID": dtxsid,
            f"min_after_{threshold_label}_filtering_corrected_conc": values_corrected.min(skipna=True),
            f"25th_percentile_after_{threshold_label}_filtering_corrected_conc": values_corrected.quantile(0.25),
            f"median_after_{threshold_label}_filtering_corrected_conc": values_corrected.median(skipna=True),
            f"mean_after_{threshold_label}_filtering_corrected_conc": values_corrected.mean(skipna=True),
            f"sd_after_{threshold_label}_filtering_corrected_conc": values_corrected.std(skipna=True, ddof=1),
            f"75th_percentile_after_{threshold_label}_filtering_corrected_conc": values_corrected.quantile(0.75),
            f"max_after_{threshold_label}_filtering_corrected_conc": values_corrected.max(skipna=True),
            f"n_nan_after_{threshold_label}_filtering": n_nan_after,
            f"n_detected_after_{threshold_label}_filtering": n_detected_after
        })

    concentration_summary_after_filtering = pd.DataFrame(after_summary_records)

    # --------------------------------------------------
    # Merge before/after summaries
    # --------------------------------------------------
    concentration_summary = concentration_summary_before_filtering.merge(
        concentration_summary_after_filtering,
        on="DTXSID",
        how="left"
    )

    concentration_summary[f"percent_detected_after_{threshold_label}_filtering"] = (
        concentration_summary[f"n_detected_after_{threshold_label}_filtering"]
        / concentration_summary["n_total_samples"]
        * 100
    )

    return (
        concentration_table_filtered_original,
        concentration_table_filtered_corrected,
        concentration_summary
    )


# ==================================================
# Step 3A. MDL filtering
# Filtering/DF based on original concentration
# Output table based on volume-corrected concentration
# ==================================================
(
    concentration_table_mdl_filtered_original,
    concentration_table_mdl_filtered_corrected,
    concentration_summary_mdl
) = filter_concentration_by_threshold(
    concentration_table_original=concentration_table_original,
    conc_cols=conc_cols,
    threshold_dict=mdl_dict,
    threshold_label="MDL",
    volume_correction_factor=VOLUME_CORRECTION_FACTOR
)

print("Final MDL-filtered ORIGINAL concentration table shape:")
print(concentration_table_mdl_filtered_original.shape)
display(concentration_table_mdl_filtered_original)

print("Final MDL-filtered VOLUME-CORRECTED concentration table shape:")
print(concentration_table_mdl_filtered_corrected.shape)
display(concentration_table_mdl_filtered_corrected)

print("MDL concentration summary:")
print(tabulate.tabulate(concentration_summary_mdl, headers="keys", tablefmt="psql"))


# ==================================================
# Step 3B. LOQ filtering
# Filtering/DF based on original concentration
# Output table based on volume-corrected concentration
# ==================================================
(
    concentration_table_loq_filtered_original,
    concentration_table_loq_filtered_corrected,
    concentration_summary_loq
) = filter_concentration_by_threshold(
    concentration_table_original=concentration_table_original,
    conc_cols=conc_cols,
    threshold_dict=LOQ_dict,
    threshold_label="LOQ",
    volume_correction_factor=VOLUME_CORRECTION_FACTOR
)

print("Final LOQ-filtered ORIGINAL concentration table shape:")
print(concentration_table_loq_filtered_original.shape)
display(concentration_table_loq_filtered_original)

print("Final LOQ-filtered VOLUME-CORRECTED concentration table shape:")
print(concentration_table_loq_filtered_corrected.shape)
display(concentration_table_loq_filtered_corrected)

print("LOQ concentration summary:")
print(tabulate.tabulate(concentration_summary_loq, headers="keys", tablefmt="psql"))

Original concentration table shape before filtering: (8, 405)


,compound_name,DTXSID,RT,Average Rt(min),Average Mz,BH04254,BH04255,BH04256,BH04257,BH04258,...,BH04644,BH04645,BH04646,BH04647,BH04648,BH04649,BH04650,BH04651,BH04652,BH04653
0,(3-Carboxypropyl)trimethylammonium chloride,DTXSID00978073,0.766,1.022,146.11743,2.926647,5.482642,10.766859,9.715496,2.197963,...,1.280863,1.43098,1.345817,1.063453,0.949321,0.962367,0.967207,1.080274,2.161299,1.083524
1,C12-ATMAC,DTXSID1026900,11.187,11.255,228.26863,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C18-BAC,DTXSID2025139,16.608,16.767,388.39389,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C12-BAC,DTXSID2040787,12.446,12.536,304.30072,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C16-BAC,DTXSID3041665,15.508,15.602,360.36395,0.198582,0.158551,0.155618,0.155618,0.271325,...,0.179361,0.18836,0.156450,0.990716,0.159558,0.454171,0.155618,0.155618,0.158276,0.158681


Final MDL-filtered ORIGINAL concentration table shape:
(8, 405)


,compound_name,DTXSID,RT,Average Rt(min),Average Mz,BH04254,BH04255,BH04256,BH04257,BH04258,...,BH04644,BH04645,BH04646,BH04647,BH04648,BH04649,BH04650,BH04651,BH04652,BH04653
0,(3-Carboxypropyl)trimethylammonium chloride,DTXSID00978073,0.766,1.022,146.11743,2.926647,5.482642,10.766859,9.715496,2.197963,...,1.280863,1.430980,1.345817,1.063453,0.949321,0.962367,0.967207,1.080274,2.161299,1.083524
1,C12-ATMAC,DTXSID1026900,11.187,11.255,228.26863,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C18-BAC,DTXSID2025139,16.608,16.767,388.39389,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C12-BAC,DTXSID2040787,12.446,12.536,304.30072,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C16-BAC,DTXSID3041665,15.508,15.602,360.36395,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.990716,NaN,0.454171,NaN,NaN,NaN,NaN
5,Choline chloride,DTXSID4020325,0.783,0.981,104.10710,57898.498833,90177.830581,152445.113191,79262.043666,53893.595800,...,39654.120682,43025.788785,12359.075103,15420.712193,14348.250632,23920.072824,33747.082952,26520.335237,50274.171068,21139.800229
6,C14-BAC,DTXSID4046606,14.059,14.269,332.33112,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,C16-ATMAC,DTXSID6026901,14.872,15.087,284.33133,3.610171,56.795919,54.299138,NaN,16.270982,...,NaN,146.159722,NaN,NaN,NaN,105.979735,NaN,NaN,NaN,NaN


Final MDL-filtered VOLUME-CORRECTED concentration table shape:
(8, 405)


...

MDL concentration summary:
+----+---------------------------------------------+----------------+--------+-------------------+--------------+-------------------------------+------------------------------------------+------------------------------------------------------+---------------------------------------------+-------------------------------------------+-----------------------------------------+------------------------------------------------------+------------------------------------------+------------------------------+------------------------------------+-------------------+----------------------------------+------------------------------------------+------------------------------------------------------+---------------------------------------------+-------------------------------------------+-----------------------------------------+------------------------------------------------------+------------------------------------------+-----------------------------+-------------------

,compound_name,DTXSID,RT,Average Rt(min),Average Mz,BH04254,BH04255,BH04256,BH04257,BH04258,...,BH04644,BH04645,BH04646,BH04647,BH04648,BH04649,BH04650,BH04651,BH04652,BH04653
0,(3-Carboxypropyl)trimethylammonium chloride,DTXSID00978073,0.766,1.022,146.11743,2.926647,5.482642,10.766859,9.715496,2.197963,...,1.280863,1.430980,1.345817,1.063453,0.949321,0.962367,0.967207,1.080274,2.161299,1.083524
1,C12-ATMAC,DTXSID1026900,11.187,11.255,228.26863,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C18-BAC,DTXSID2025139,16.608,16.767,388.39389,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C12-BAC,DTXSID2040787,12.446,12.536,304.30072,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C16-BAC,DTXSID3041665,15.508,15.602,360.36395,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Choline chloride,DTXSID4020325,0.783,0.981,104.10710,57898.498833,90177.830581,152445.113191,79262.043666,53893.595800,...,39654.120682,43025.788785,12359.075103,15420.712193,14348.250632,23920.072824,33747.082952,26520.335237,50274.171068,21139.800229
6,C14-BAC,DTXSID4046606,14.059,14.269,332.33112,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,C16-ATMAC,DTXSID6026901,14.872,15.087,284.33133,3.610171,56.795919,54.299138,NaN,16.270982,...,NaN,146.159722,NaN,NaN,NaN,105.979735,NaN,NaN,NaN,NaN


Final LOQ-filtered VOLUME-CORRECTED concentration table shape:
(8, 405)


,compound_name,DTXSID,RT,Average Rt(min),Average Mz,BH04254,BH04255,BH04256,BH04257,BH04258,...,BH04644,BH04645,BH04646,BH04647,BH04648,BH04649,BH04650,BH04651,BH04652,BH04653
0,(3-Carboxypropyl)trimethylammonium chloride,DTXSID00978073,0.766,1.022,146.11743,1.755988,3.289585,6.460116,5.829298,1.318778,...,0.768518,0.858588,0.807490,0.638072,0.569592,0.577420,0.580324,0.648164,1.296780,0.650115
1,C12-ATMAC,DTXSID1026900,11.187,11.255,228.26863,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C18-BAC,DTXSID2025139,16.608,16.767,388.39389,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C12-BAC,DTXSID2040787,12.446,12.536,304.30072,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C16-BAC,DTXSID3041665,15.508,15.602,360.36395,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Choline chloride,DTXSID4020325,0.783,0.981,104.10710,34739.099300,54106.698349,91467.067915,47557.226200,32336.157480,...,23792.472409,25815.473271,7415.445062,9252.427316,8608.950379,14352.043694,20248.249771,15912.201142,30164.502641,12683.880138
6,C14-BAC,DTXSID4046606,14.059,14.269,332.33112,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,C16-ATMAC,DTXSID6026901,14.872,15.087,284.33133,2.166103,34.077551,32.579483,NaN,9.762589,...,NaN,87.695833,NaN,NaN,NaN,63.587841,NaN,NaN,NaN,NaN


LOQ concentration summary:
+----+---------------------------------------------+----------------+--------+-------------------+--------------+-------------------------------+------------------------------------------+------------------------------------------------------+---------------------------------------------+-------------------------------------------+-----------------------------------------+------------------------------------------------------+------------------------------------------+------------------------------+------------------------------------+-------------------+----------------------------------+------------------------------------------+------------------------------------------------------+---------------------------------------------+-------------------------------------------+-----------------------------------------+------------------------------------------------------+------------------------------------------+-----------------------------+-------------------

In [64]:
# save output
# output_path = r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QAcompound_estimated_concentration_MDL_20260427.csv'
output_path = r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QAcompound_estimated_concentration_MDL_20260429.csv'
output_path2 = r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QAcompound_summary_estimated_concentration_MDL_20260429.csv'
concentration_table_mdl_filtered_corrected.to_csv(output_path, index=False)
concentration_summary_mdl.to_csv(output_path2, index = False)

print(f"Estimated concentration file saved to:\n{output_path}")

Estimated concentration file saved to:
D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QAcompound_estimated_concentration_MDL_20260429.csv


In [ ]:
concentration_table_mdl_filtered_corrected = pd.read_csv(r'D:\UCSF_postdoc_topic\Collaboration\prenatal_exposure_QAC\QAcompound_estimated_concentration_MDL_20260429.csv')
print(tabulate.tabulate(concentration_table_mdl_filtered_corrected,headers='keys', tablefmt='psql'))

+----+---------------------------------------------+----------------+--------+-------------------+--------------+-------------+-------------+-------------+------------+-------------+-------------+--------------+-------------+-------------+-------------+-------------+-------------+-------------+--------------+-------------+--------------+--------------+--------------+-------------+--------------+--------------+--------------+--------------+-------------+-------------+-------------+------------+--------------+-------------+--------------+--------------+--------------+------------+--------------+-------------+--------------+--------------+-------------+--------------+--------------+--------------+-------------+-------------+--------------+---------------+-------------+-------------+-------------+-------------+--------------+-------------+-------------+--------------+-------------+-------------+-------------+-------------+--------------+-------------+-------------+--------------+----------